# Pelatihan Model Bahasa Isyarat & Ekspor ke TensorFlow.js
Notebook ini membaca `dataset_landmarks.csv`, melatih Multi-Layer Perceptron (MLP), dan mengekspor hasilnya menjadi `model.json` dan `group1-shard1of1.bin`.

In [ ]:
!pip install tensorflowjs pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import json
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflowjs as tfjs

# 1. Muat dataset
df = pd.read_csv('dataset_landmarks.csv')
X = df.iloc[:, :-1].values.astype(np.float32)
y_raw = df.iloc[:, -1].values

# 2. Encode label ke one-hot tensor
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_raw)
y = tf.keras.utils.to_categorical(y_encoded)
num_classes = len(encoder.classes_)

# Simpan label mapping
labels_dict = {int(i): str(label) for i, label in enumerate(encoder.classes_)}
with open('labels.json', 'w') as f:
    json.dump(labels_dict, f, indent=2)

print(f"Dataset loaded: {X.shape[0]} samples, {num_classes} classes: {encoder.classes_}")

In [ ]:
# 3. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Arsitektur Jaringan Neural (MLP)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(63,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# 5. Pelatihan model
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1
)

In [ ]:
# 6. Ekspor ke TensorFlow.js format
!mkdir -p model_web
tfjs.converters.save_keras_model(model, 'model_web')
!cp labels.json model_web/

# Kompres dan unduh berkas zip
!zip -r model_web.zip model_web
from google.colab import files
files.download('model_web.zip')